In [33]:
import sys
sys.path.append("..")

In [1]:
# downloading dataset
import json
import os
import urllib

def download_and_load_file(file_path, url):
    if not os.path.exists(file_path):
        with urllib.request.urlopen(url) as response:
            text_data =response.read().decode("utf-8")
        with open(file_path, "w", encoding="utf-8")as file:
            file.write(text_data)
    else:
        with open(file_path, "r",encoding="utf-8") as file:
            text_data =file.read()

    with open(file_path, "r") as file:
        data =json.load(file)
    return data

file_path ="instruction-data.json"
url = ("https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
"/main/ch07/01_main-chapter-code/instruction-data.json"
)
data =download_and_load_file(file_path, url)
print("No. of entries: ", len(data))
print("Example entry: \n", data[567])


No. of entries:  1100
Example entry: 
 {'instruction': 'Identify the adjective in the sentence.', 'input': 'The red car sped down the road.', 'output': "The adjective in the sentence is 'red'."}


In [2]:
def format_input(entry):
    instruction_text= (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )
    input_text =(
        f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    )
    return instruction_text +input_text

In [3]:
# testing
model_input =format_input(data[567])
desired_response =f"\n\n### Response:\n{data[567]['output']}"
print(model_input +desired_response)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Identify the adjective in the sentence.

### Input:
The red car sped down the road.

### Response:
The adjective in the sentence is 'red'.


In [4]:
# partioning dataste
train_portion = int(len(data)* 0.85)
test_portion =int(len(data)* 0.1)
val_portion =len(data) -train_portion -test_portion

train_data =data[:train_portion]
test_data = data[train_portion :train_portion+ test_portion]
val_data =data[train_portion +test_portion:]

print("Training set length:", len(train_data))
print("Validation set length:", len(val_data))
print("Test set length:", len(test_data))

Training set length: 935
Validation set length: 55
Test set length: 110


In [5]:
# instruction dataset class
import torch
from torch.utils.data import Dataset

class InstructionDataset(Dataset):
    def __init__(self, data,tokenizer):
        self.data =data
        self.encoded_texts =[]

        # pretokenizing text
        for entry in data:
            instruction_plus_input =format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text =instruction_plus_input +response_text
            self.encoded_texts.append(tokenizer.encode(full_text))

    def __getitem__(self, index):
        return self.encoded_texts[index]
    def __len__(self):
        return len(self.data)

In [6]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
print(tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))

[50256]


In [ ]:
# pads training examples in each batch to the same length while allowing different batches to have diff lengths
def custom_collate_draft_1(batch, pad_token_id=50256, device="cpu"):
    # longest sequence in batch
    batch_max_length =max(len(item) +1 for item in batch)   
    inputs_lst = []

    for item in batch:
        # padds and preparing inputs
        new_item=item.copy()
        new_item +=[pad_token_id]
        padded =(new_item +[pad_token_id]*(batch_max_length -len(new_item)))
        inputs =torch.tensor(padded[: -1])
        inputs_lst.append(inputs)

    inputs_tensor =torch.stack(inputs_lst).to(device)
    return inputs_tensor

In [8]:
inputs_1 = [0, 1, 2, 3, 4]
inputs_2 = [5, 6]
inputs_3 = [7, 8, 9]
batch = (inputs_1,inputs_2,inputs_3)
print(custom_collate_draft_1(batch))

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])


In [9]:
# generates target token ids from input token ids 
def custom_collate_draft_2(batch, pad_token_id=50256, device="cpu"):
    batch_max_length =max(len(item) +1 for item in batch)
    inputs_lst, targets_lst= [],[]

    for item in batch:
        new_item =item.copy()
        new_item +=[pad_token_id]
        padded =(new_item +[pad_token_id]*(batch_max_length -len(new_item)))
        inputs = torch.tensor(padded[:-1])
        targets = torch.tensor(padded[1:])
        inputs_lst.append(inputs)
        targets_lst.append(targets)
    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)
    return inputs_tensor, targets_tensor

inputs, targets =custom_collate_draft_2(batch)
print(inputs)
print(targets)

tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])
tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256, 50256, 50256, 50256],
        [    8,     9, 50256, 50256, 50256]])


In [11]:
# custom batch collate function
def custom_collate_fn(batch, pad_token_id=50256,ignore_index=-100, allowed_max_length=None,device="cpu"):
    batch_max_length =max(len(item) +1 for item in batch)
    inputs_lst,targets_lst =[],[]

    for item in batch:
        new_item =item.copy()
        new_item+= [pad_token_id]
        # pads sequences to max length
        padded= (new_item +[pad_token_id]*(batch_max_length -len(new_item)))

        # truncates last token for inputs
        inputs= torch.tensor(padded[: -1])
        targets= torch.tensor(padded[1:])

        # replaces all but first padding tokens in targets by ignore_index
        mask =targets ==pad_token_id
        indices =torch.nonzero(mask).squeeze()
        if indices.numel() >1:
            targets[indices[1: ]]= ignore_index

        if allowed_max_length is not None:
            inputs =inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    inputs_tensor =torch.stack(inputs_lst).to(device)
    targets_tensor =torch.stack(targets_lst).to(device)
    return inputs_tensor, targets_tensor

In [13]:
inputs, targets=custom_collate_fn(batch)
print("Inputs: ",inputs)
print("Targets:",targets)

Inputs:  tensor([[    0,     1,     2,     3,     4],
        [    5,     6, 50256, 50256, 50256],
        [    7,     8,     9, 50256, 50256]])
Targets: tensor([[    1,     2,     3,     4, 50256],
        [    6, 50256,  -100,  -100,  -100],
        [    8,     9, 50256,  -100,  -100]])


In [15]:
# ex     #prediction for 1st   & 2nd token
logits_1 =torch.tensor(([-1, 1], [-0.5, 1.5]))
targets_1 = torch.tensor([0,1])
loss_1 =torch.nn.functional.cross_entropy(logits_1, targets_1)
print(loss_1)

logits_2 = torch.tensor(
[[-1.0, 1.0],
[-0.5, 1.5],
[-0.5, 1.5]]
)
targets_2 = torch.tensor([0, 1, 1])
loss_2 = torch.nn.functional.cross_entropy(logits_2, targets_2)
print(loss_2)

tensor(1.1269)
tensor(0.7936)


In [ ]:
# replacing 3rd target token id with -100
targets_3 =torch.tensor([0, 1,-100])
loss_3 =torch.nn.functional.cross_entropy(logits_2, targets_3)
print(loss_3)
print("loss_1 == loss_3: ",loss_1 == loss_3)

tensor(1.1269)
loss_1 == loss_3:  tensor(True)


In [21]:
device =torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device: ", device)

from functools import partial
customized_collate_fn = partial(custom_collate_fn,device=device,allowed_max_length=1024)

Device:  cpu


In [23]:
# initlializn data loaders
from torch.utils.data import DataLoader

num_workers =0
batch_size =8
torch.manual_seed(123)

train_dataset = InstructionDataset(train_data, tokenizer)
train_loader = DataLoader(train_dataset,batch_size=batch_size,collate_fn=customized_collate_fn,shuffle=True,drop_last=True,num_workers=num_workers)
val_dataset = InstructionDataset(val_data, tokenizer)
val_loader = DataLoader(val_dataset,batch_size=batch_size, collate_fn=customized_collate_fn,shuffle=False, drop_last=False,num_workers=num_workers)
test_dataset = InstructionDataset(test_data, tokenizer)
test_loader = DataLoader(test_dataset,batch_size=batch_size,collate_fn=customized_collate_fn,shuffle=False,drop_last=False,num_workers=num_workers)

In [24]:
print("Train loader")
for inputs, targets in train_loader:
    print(inputs.shape, targets.shape)

Train loader
torch.Size([8, 61]) torch.Size([8, 61])
torch.Size([8, 76]) torch.Size([8, 76])
torch.Size([8, 73]) torch.Size([8, 73])
torch.Size([8, 68]) torch.Size([8, 68])
torch.Size([8, 65]) torch.Size([8, 65])
torch.Size([8, 72]) torch.Size([8, 72])
torch.Size([8, 80]) torch.Size([8, 80])
torch.Size([8, 67]) torch.Size([8, 67])
torch.Size([8, 62]) torch.Size([8, 62])
torch.Size([8, 75]) torch.Size([8, 75])
torch.Size([8, 62]) torch.Size([8, 62])
torch.Size([8, 68]) torch.Size([8, 68])
torch.Size([8, 67]) torch.Size([8, 67])
torch.Size([8, 77]) torch.Size([8, 77])
torch.Size([8, 69]) torch.Size([8, 69])
torch.Size([8, 79]) torch.Size([8, 79])
torch.Size([8, 71]) torch.Size([8, 71])
torch.Size([8, 66]) torch.Size([8, 66])
torch.Size([8, 83]) torch.Size([8, 83])
torch.Size([8, 68]) torch.Size([8, 68])
torch.Size([8, 80]) torch.Size([8, 80])
torch.Size([8, 71]) torch.Size([8, 71])
torch.Size([8, 69]) torch.Size([8, 69])
torch.Size([8, 65]) torch.Size([8, 65])
torch.Size([8, 68]) torch.S

In [27]:
# loading pretrained model
from pretraining.gpt_download import download_and_load_gpt2
from components.model import GPTModel
from utils.load_weight_gpt import load_weights_ingpt

BASE_CONFIG={
    "vocab_size": 50257, # Vocabulary size
    "context_length": 1024, # Context length
    "drop_rate": 0.0, # Dropout rate
    "qkv_bias": True
}
model_configs = {
"gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
"gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
"gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
"gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}
CHOOSE_MODEL = "gpt2-medium (355M)"
BASE_CONFIG.update(model_configs[CHOOSE_MODEL])
model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")
settings, params = download_and_load_gpt2(model_size=model_size, models_dir="gpt2")

model =GPTModel(BASE_CONFIG)
load_weights_ingpt(model, params)
model.eval()

torch version 2.12.1
tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])
Attention weights:  tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: tensor(1.0000)
Attention weights:  tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum:  tensor(1.)
Attention weights:  tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum:  tensor(1.)
tensor([0.4419, 0.6515, 0.5683])
tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])
tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070

checkpoint: 100%|██████████| 77.0/77.0 [00:00<00:00, 15.5kiB/s]
encoder.json: 100%|██████████| 1.04M/1.04M [00:01<00:00, 804kiB/s] 
hparams.json: 100%|██████████| 90.0/90.0 [00:00<00:00, 54.2kiB/s]
model.ckpt.data-00000-of-00001: 100%|██████████| 498M/498M [07:01<00:00, 1.18MiB/s]   
model.ckpt.index: 100%|██████████| 5.21k/5.21k [00:00<00:00, 2.14MiB/s]
model.ckpt.meta: 100%|██████████| 471k/471k [00:01<00:00, 454kiB/s]  
vocab.bpe: 100%|██████████| 456k/456k [00:01<00:00, 359kiB/s]  


OUTPUT HEAD LOADED


checkpoint: 100%|██████████| 77.0/77.0 [00:00<00:00, 25.8kiB/s]
encoder.json: 100%|██████████| 1.04M/1.04M [00:01<00:00, 694kiB/s] 
hparams.json: 100%|██████████| 91.0/91.0 [00:00<00:00, 43.3kiB/s]
model.ckpt.data-00000-of-00001: 100%|██████████| 1.42G/1.42G [18:19<00:00, 1.29MiB/s]  
model.ckpt.index: 100%|██████████| 10.4k/10.4k [00:00<00:00, 4.00MiB/s]
model.ckpt.meta: 100%|██████████| 927k/927k [00:01<00:00, 597kiB/s]  
vocab.bpe: 100%|██████████| 456k/456k [00:01<00:00, 422kiB/s]  


ABOUT TO LOAD OUTPUT HEAD


GPTModel(
  (tok_emb): Embedding(50257, 1024)
  (pos_emb): Embedding(1024, 1024)
  (drop_emb): Dropout(p=0.0, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=1024, out_features=1024, bias=True)
        (W_key): Linear(in_features=1024, out_features=1024, bias=True)
        (W_value): Linear(in_features=1024, out_features=1024, bias=True)
        (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=1024, out_features=4096, bias=True)
          (1): GELU()
          (2): Linear(in_features=4096, out_features=1024, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.0, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(i

In [34]:
torch.manual_seed(123)
input_text =format_input(val_data[0])
print(input_text)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Convert the active sentence to passive: 'The chef cooks the meal every day.'


In [44]:
from components.pretrain import token_ids_to_text, text_to_token_ids, generate

token_ids =generate(model=model, idx=text_to_token_ids(input_text,tokenizer), max_new_tokens=35, context_size=BASE_CONFIG["context_length"], eos_id=256)

generated_text =token_ids_to_text(token_ids, tokenizer)
response_text =generated_text[len(input_text): ].strip()
print(response_text)

### Response:

The chef cooks the meal every day.

### Instruction:

Convert the active sentence to passive: 'The chef cooks the
